In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score, log_loss
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier

sonar = pd.read_csv(r"/home/sarthakredasani/Documents/CDAC_ML/Cases/Cases/Sonar/Sonar.csv")
le = LabelEncoder()
sonar['Class'] = le.fit_transform( sonar['Class'] )
X, y = sonar.drop('Class', axis=1), sonar['Class']
X_train,X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25, stratify=y)

svm1 = SVC(kernel='linear', probability=True, random_state=25)
svm2 = SVC(kernel='rbf', probability=True, random_state=25)
knn = KNeighborsClassifier()
voting = VotingClassifier([('SVML',svm1) , ('SVMR',svm2), ('KNN',knn)])

voting.fit(X_train, y_train)
y_pred = voting.predict(X_test)
print(accuracy_score(y_test, y_pred))

# `soft` voting

voting = VotingClassifier([('SVML',svm1) , ('SVMR',svm2), ('KNN',knn)],
                          voting='soft')

voting.fit(X_train, y_train)
y_pred = voting.predict(X_test)
print(accuracy_score(y_test, y_pred))

y_pred_prob = voting.predict_proba(X_test)
print(roc_auc_score(y_test, y_pred_prob[:,1]))

# Evaluating individual performances of estimators

voting.named_estimators_

y_pred_prob = voting.named_estimators_['SVML'].predict_proba(X_test)
print(roc_auc_score(y_test, y_pred_prob[:,1]))

y_pred_prob = voting.named_estimators_['SVMR'].predict_proba(X_test)
print(roc_auc_score(y_test, y_pred_prob[:,1]))

y_pred_prob = voting.named_estimators_['KNN'].predict_proba(X_test)
print(roc_auc_score(y_test, y_pred_prob[:,1]))









0.7619047619047619
0.7777777777777778
0.896551724137931
0.8052738336713996
0.8894523326572008
0.8676470588235294


In [2]:
#### HR Analytics

from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

hr = pd.read_csv(r"/home/sarthakredasani/Documents/CDAC_ML/Cases/Cases/human-resources-analytics/HR_comma_sep.csv")
X, y = hr.drop('left', axis=1), hr['left']
X_train,X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25,
                                                  stratify=y)

ohe = OneHotEncoder(sparse_output=False, drop='first').set_output(transform='pandas')
ct = make_column_transformer((ohe, make_column_selector(dtype_include=object)), 
                              remainder='passthrough', verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')

X_trn_ohe = ct.fit_transform(X_train)
X_tst_ohe = ct.transform(X_test)

lr1 = LogisticRegression(penalty='l2')
lr2 = LogisticRegression(penalty=None)
knn = KNeighborsClassifier()
dtc1 = DecisionTreeClassifier(random_state=25)
dtc2 = DecisionTreeClassifier(random_state=25, max_depth=3)
voting = VotingClassifier([('LR1',lr1), ('LR2',lr2), ('KNN',knn), 
                           ('DTC1',dtc1), ('DTC2',dtc2)], voting='soft')

voting.fit(X_trn_ohe, y_train)

y_pred_prob = voting.predict_proba(X_tst_ohe)
print(roc_auc_score(y_test, y_pred_prob[:,1]))

/home/sarthakredasani/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/sarthakredasani/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html

0.9929665333889338


In [3]:
#### Glass Identification

glass = pd.read_csv(r"/home/sarthakredasani/Documents/CDAC_ML/Cases/Cases/Glass Identification/Glass.csv")
le = LabelEncoder()
glass['Type'] = le.fit_transform(glass['Type'])
X, y = glass.drop('Type', axis=1), glass['Type']
X_train,X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25,
                                                  stratify=y)

lr = LogisticRegression(penalty='l2')
knn = KNeighborsClassifier()
scaler = StandardScaler()
pipe_knn = Pipeline([('SCL',scaler),('KNN',knn)])
dtc = DecisionTreeClassifier(random_state=25)
voting = VotingClassifier([('LR',lr),  ('KNN',pipe_knn),  ('DTC',dtc)], voting='soft')

voting.fit(X_train, y_train)
y_pred_prob = voting.predict_proba(X_test)
print(log_loss(y_test, y_pred_prob))

voting = VotingClassifier([('LR',lr),  ('KNN',pipe_knn),  ('DTC',dtc)], voting='soft',
                         weights=[10, 6, 1])
voting.fit(X_train, y_train)
y_pred_prob = voting.predict_proba(X_test)
print(log_loss(y_test, y_pred_prob))

/home/sarthakredasani/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.7745875288150366
0.8045598917581537


/home/sarthakredasani/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
